# SV1 report — Simt + handwritten SIMD DSL

Interactive companion to `ANALYSIS.md`.

**SOC:** Ascend950PR_9599 opsim · **Focus:** simt-asc / simd-pto / handwritten SIMD DSL

This notebook:
1. Loads `config.toml` case list
2. Regenerates Simt sensitivity plots from `data/SUMMARY_sv1_simt_tile16k.tsv` (offline-safe)
3. Overlays SIMD DSL when `data/SUMMARY_sv1d_*.tsv` exists
4. Documents rooftop (32 vs 64 el/cyc) and optional opsim re-run cells

**Offline mode:** plot cells only need pandas / matplotlib + the TSVs under `data/` — no CANN.


In [ ]:
from pathlib import Path
import sys

HERE = Path.cwd().resolve()
# Prefer running with kernel CWD = benches/sv1 (or notebooks next to data/)
CANDIDATES = [HERE, HERE / "benches" / "sv1", HERE.parent / "benches" / "sv1"]
ROOT = next((p for p in CANDIDATES if (p / "config.toml").exists() or (p / "data").exists()), HERE)
DATA = ROOT / "data"
PLOTS = ROOT / "plots"
PLOTS.mkdir(exist_ok=True)
print("ROOT =", ROOT)
print("DATA exists:", DATA.exists(), "| files:", sorted(p.name for p in DATA.glob("*")) if DATA.exists() else [])


In [ ]:
# Optional deps — notebook stays usable if tomllib/pandas missing for some cells
import importlib

def need(pkg, pip_name=None):
    try:
        return importlib.import_module(pkg)
    except ImportError as e:
        print(f"MISSING {pkg}: {e}\n  pip install {pip_name or pkg}")
        return None

pd = need("pandas")
plt_mod = need("matplotlib.pyplot", "matplotlib")
np = need("numpy")
try:
    import tomllib
except ModuleNotFoundError:
    tomllib = need("tomli", "tomli")
    if tomllib:
        import tomli as tomllib

print("pandas", bool(pd), "matplotlib", bool(plt_mod), "tomllib", bool(tomllib))


## 1. Case list from `config.toml`

In [ ]:
cfg = {}
cases = []
if tomllib and (ROOT / "config.toml").exists():
    cfg = tomllib.loads((ROOT / "config.toml").read_text())
    cases = cfg.get("cases", [])
    print(f"id={cfg.get('id')} title={cfg.get('title')}")
    print(f"focus={cfg.get('focus')}")
    print(f"n_cases={len(cases)}")
else:
    print("config.toml not loaded — skip table")

if pd and cases:
    rows = []
    for c in cases:
        sh = c.get("shape") or {}
        rows.append({
            "tag": c.get("tag"),
            "suite": c.get("suite"),
            "path": c.get("path", "simt-asc"),
            **{k: sh.get(k) for k in ("E", "T", "TILE", "VL", "ept")},
        })
    display(pd.DataFrame(rows))
elif cases:
    for c in cases[:15]:
        print(c.get("tag"), c.get("suite"), c.get("shape"))
    if len(cases) > 15:
        print(f"... +{len(cases)-15} more")


## 2. Load Simt E=16K harvest

In [ ]:
SIMT_TSV = DATA / "SUMMARY_sv1_simt_tile16k.tsv"
assert SIMT_TSV.exists(), f"missing {SIMT_TSV}"
df = pd.read_csv(SIMT_TSV, sep="\t")
# numeric cleanup
for col in ["E", "TILE", "T", "ept", "us", "VF_cyc", "elems_per_cyc", "LDS", "FADD", "STS",
            "IADD_tot", "IADD_over_memF", "IADD_over_body", "IPC_body"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
print(df.shape)
display(df.sort_values(["T", "TILE"]).reset_index(drop=True))


## 3. TILE @ T=128 — VF cycles / el/cyc / instr

In [ ]:
import matplotlib.pyplot as plt

tile = df[(df["T"] == 128) & (df["tag"].str.startswith("sv1tt_"))].sort_values("TILE")
assert not tile.empty, "no sv1tt T=128 rows"

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(tile["TILE"], tile["VF_cyc"], "o-", color="#1f77b4")
axes[0].set_xlabel("TILE"); axes[0].set_ylabel("VF cycles"); axes[0].set_title("TILE vs VF_cyc @ T128")
axes[0].grid(True, alpha=0.3)

axes[1].plot(tile["TILE"], tile["elems_per_cyc"], "o-", color="#2ca02c")
axes[1].axhline(32, ls="--", color="gray", label="Simt roof 32 el/cyc")
axes[1].axhline(64, ls=":", color="orange", label="SIMD UB roof ~64")
axes[1].set_xlabel("TILE"); axes[1].set_ylabel("el/cyc"); axes[1].set_title("TILE vs el/cyc")
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

body = tile["LDS"] + tile["FADD"] + tile["STS"] + tile["IADD_tot"]
axes[2].plot(tile["TILE"], body, "o-", label="body (LDS+FADD+STS+IADD)")
axes[2].plot(tile["TILE"], tile["IADD_tot"], "s--", label="IADD_tot")
axes[2].set_xlabel("TILE"); axes[2].set_ylabel("instr count"); axes[2].set_title("TILE vs instr")
axes[2].legend(fontsize=8); axes[2].grid(True, alpha=0.3)

fig.tight_layout()
out = PLOTS / "nb_tile_vs_vf_el_instr.png"
fig.savefig(out, dpi=120, bbox_inches="tight")
print("saved", out)
print(tile[["TILE", "ept", "VF_cyc", "elems_per_cyc", "IADD_tot", "IPC_body", "pack"]].to_string(index=False))
plt.show()


## 4. Untiled thread sweep — IPC / el/cyc

In [ ]:
thr = df[df["tag"].str.startswith("sv1t_")].sort_values("T")
assert not thr.empty

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(thr["T"], thr["IPC_body"], "o-", color="#d62728")
axes[0].set_xlabel("T"); axes[0].set_ylabel("IPC_body"); axes[0].set_title("T vs IPC @ E=16K untiled")
axes[0].grid(True, alpha=0.3)

axes[1].plot(thr["T"], thr["elems_per_cyc"], "o-", color="#2ca02c")
axes[1].axhline(32, ls="--", color="gray", label="Simt roof 32")
axes[1].set_xlabel("T"); axes[1].set_ylabel("el/cyc"); axes[1].set_title("T vs el/cyc")
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

fig.tight_layout()
out = PLOTS / "nb_thread_vs_ipc_el.png"
fig.savefig(out, dpi=120, bbox_inches="tight")
print("saved", out)
print(thr[["T", "VF_cyc", "elems_per_cyc", "IADD_tot", "IPC_body"]].to_string(index=False))
plt.show()


## 5. Large-tile instruction breakdown

In [ ]:
# Prefer TILE=8192 @ T128; fall back to best TILE=2048
big = tile[tile["TILE"] == 8192]
if big.empty:
    big = tile[tile["TILE"] == 2048]
row = big.iloc[0]
labels = ["LDS", "FADD", "STS", "IADD_tot"]
vals = [row[c] for c in labels]
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(labels, vals, color=["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"])
ax.set_title(f"Instr breakdown {row['tag']}")
ax.set_ylabel("call_count")
for i, v in enumerate(vals):
    ax.text(i, v, f"{int(v)}", ha="center", va="bottom")
fig.tight_layout()
out = PLOTS / "nb_large_tile_instr_breakdown.png"
fig.savefig(out, dpi=120, bbox_inches="tight")
print("saved", out, "pack=", row.get("pack"), "ratio=", row.get("ratio"))
plt.show()


## 6. SIMD DSL overlay (when SUMMARY present)

Looks for `data/SUMMARY_sv1d_size_ladder.tsv` or `data/SUMMARY_sv1d_cost.tsv`.
If missing / all FAIL, plots Simt-only and shows a stub table for E≥8K remeasure.


In [ ]:
simd_paths = [
    DATA / "SUMMARY_sv1d_size_ladder.tsv",
    DATA / "SUMMARY_sv1d_cost.tsv",
]
simd_path = next((p for p in simd_paths if p.exists()), None)
df_simd = None
if simd_path:
    df_simd = pd.read_csv(simd_path, sep="\t")
    print("loaded", simd_path.name, "rows=", len(df_simd))
    # normalize columns
    rename = {}
    if "VF_cycles" in df_simd.columns and "VF_cyc" not in df_simd.columns:
        rename["VF_cycles"] = "VF_cyc"
    df_simd = df_simd.rename(columns=rename)
    for col in ["E", "VL", "VF_cyc", "body_instr", "IPC", "vecs", "cyc_per_vec", "us"]:
        if col in df_simd.columns:
            df_simd[col] = pd.to_numeric(df_simd[col], errors="coerce")
    if "elems_per_cyc" not in df_simd.columns and "VF_cyc" in df_simd.columns and "E" in df_simd.columns:
        df_simd["elems_per_cyc"] = df_simd["E"] / df_simd["VF_cyc"]
    display(df_simd)
else:
    print("No SUMMARY_sv1d_*.tsv under data/ — stub only.")
    print("Drop a ladder TSV here after E≥8K opsim, then re-run this cell.")


In [ ]:
# Overlay: Simt best el/cyc by TILE (proxy size) vs SIMD el/cyc by E
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(tile["TILE"], tile["elems_per_cyc"], "o-", label="Simt TILE@T128 (E=16K)")
ax.axhline(32, ls="--", color="gray", alpha=0.7, label="Simt roof 32 el/cyc")
ax.axhline(64, ls=":", color="orange", alpha=0.8, label="SIMD UB roof ~64 el/cyc")

if df_simd is not None:
    ok = df_simd.copy()
    if "status" in ok.columns:
        ok = ok[ok["status"].astype(str).str.upper() == "PASS"]
    ok = ok.dropna(subset=["VF_cyc", "E"]) if "VF_cyc" in ok.columns else ok.iloc[0:0]
    if not ok.empty:
        ok = ok.sort_values("E")
        ax.plot(ok["E"], ok["elems_per_cyc"], "s-", color="#9467bd", label="SIMD DSL (E ladder)")
        print(ok[["tag", "E", "VF_cyc", "elems_per_cyc"]].to_string(index=False))
    else:
        print("SIMD TSV present but no PASS rows with VF_cyc — stub overlay only.")
else:
    # Known small-E footnote from prior harvest (E=256 ≈ 59 VF cyc → ~4.3 el/cyc)
    ax.scatter([256], [256/59], marker="x", color="#9467bd", s=80, label="SIMD DSL E=256 footnote (~4.3)")
    print("Stub: only E=256 footnote (~59 VF cyc). Need E≥8K ladder for asymptotic el/cyc.")

ax.set_xscale("log", base=2)
ax.set_xlabel("TILE or E")
ax.set_ylabel("el/cyc")
ax.set_title("Size / tile vs el/cyc — Simt + SIMD DSL")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
fig.tight_layout()
out = PLOTS / "nb_simt_simd_el_overlay.png"
fig.savefig(out, dpi=120, bbox_inches="tight")
print("saved", out)
plt.show()


## 7. Rooftop notes — LDS/STS.64 vs .128 · 32 vs 64 el/cyc

| Path | Width | Theory roof | Measured peak (E=16K) | Gap |
|------|-------|-------------|------------------------|-----|
| Simt-asc | LDS/STS.**64** (float2); no .128 | **32 el/cyc** if IPC≈4 + FADD=32-el/issue | **3.71** (untiled T512) / **3.22** (TILE2048@T128) | ~9–10× |
| Handwritten SIMD DSL | VL=64 (`2×vload + vadd + vstore`) | **~64 el/cyc** if α→1 cyc/vec (UB-bound) | E=256 only ≈**4.3**; E≥8K **TBD** | overhead / unmeasured |

**Todos (from exploration):** LDS/STS.128 (float4); imm-offset LDS/STS (unroll, cut IADD); large-T vs T=128 with unroll + scheduling; measure SIMD asymptotic α at E≥8K.

See `data/SIMT_LDST_WIDTH_20261008.md` and `ANALYSIS.md` for full math.


In [ ]:
# Quick rooftop gap table from current Simt harvest
points = []
for label, sub in [
    ("TILE2048@T128", tile[tile["TILE"] == 2048]),
    ("untiled T512", thr[thr["T"] == 512]),
]:
    if sub.empty:
        continue
    r = sub.iloc[0]
    points.append({
        "point": label,
        "el/cyc": round(float(r["elems_per_cyc"]), 3),
        "IPC_body": round(float(r["IPC_body"]), 3),
        "roof_el": 32,
        "gap_el": round(32 / float(r["elems_per_cyc"]), 1),
        "IPC_needed_~4": 4.0,
        "gap_ipc": round(4.0 / float(r["IPC_body"]), 1),
    })
display(pd.DataFrame(points))


## 8. Optional — re-run one opsim tag (pto-b10)

Does **not** hard-fail if CANN / oneshot scripts are missing. Set env vars, pick a tag, then run.

Expected on login node (see tilelang-ascend-env skill):
- `PY` = `~/projects/tilelang-deepseek/.venv-npu/bin/python`
- `ASCEND_HOME_PATH` = `cann_91b3` (camodel) or suite pin
- `TORCH_DEVICE_BACKEND_AUTOLOAD=0`
- suite oneshots under gym root or `/tmp/sv1_cost_sweep`


In [ ]:
import os, shutil, subprocess

# Edit these then run
TAG = os.environ.get("SV1_RERUN_TAG", "sv1tt_e16384_tile2048_t128")
ONESHOT = Path(os.environ.get(
    "SV1_ONESHOT",
    str(ROOT.parents[1] / "oneshot_sv1_simt_tile16k.sh") if (ROOT.parents[1] / "oneshot_sv1_simt_tile16k.sh").exists()
    else "/tmp/sv1_cost_sweep/oneshot_sv1_simt_tile16k.sh",
))
DRY_RUN = True  # set False to actually invoke

print("TAG =", TAG)
print("ONESHOT =", ONESHOT, "exists=", ONESHOT.exists())
print("ASCEND_HOME_PATH =", os.environ.get("ASCEND_HOME_PATH", "<unset>"))
print("PYTHON_BIN =", os.environ.get("PYTHON_BIN", shutil.which("python3")))
print("TORCH_DEVICE_BACKEND_AUTOLOAD =", os.environ.get("TORCH_DEVICE_BACKEND_AUTOLOAD", "<unset>"))

missing = []
if not ONESHOT.exists():
    missing.append("oneshot script")
if not os.environ.get("ASCEND_HOME_PATH"):
    missing.append("ASCEND_HOME_PATH")
if missing:
    print("SKIP opsim — missing:", ", ".join(missing))
    print("Documented env only; plots above remain valid offline.")
else:
    cmd = ["bash", str(ONESHOT)]
    print("cmd:", cmd, "| DRY_RUN=", DRY_RUN)
    if not DRY_RUN:
        subprocess.run(cmd, check=False)


## 9. How to restart this notebook server

See `JUPYTER.md` in this folder. Typical:

```bash
cd ~/projects/tilelangGym/benches/sv1
source ~/projects/tilelang-deepseek/.venv-npu/bin/activate   # or dedicated venv
jupyter lab --no-browser --ip=0.0.0.0 --port=8899 --ServerApp.token=... 
```

From Windows DESKTOP-V7R2A6I:

```powershell
ssh -L 8899:127.0.0.1:8899 pto-b10
# then open http://127.0.0.1:8899/lab?token=...
```
